# BN_C — Data Pipeline and Monte Carlo Vertex Search

This notebook is the **real-data entry point**: it reads  files, builds the  (modal values) and  (full triangle PDFs) via convolution, then runs Monte Carlo and simulated-annealing search to find candidate Φ solutions. For provably optimal solutions (exact sup-norm LP, L2/QP), see **Notebook B**.

Source: .

### File path here:

In [3]:
import numpy as np
import matplotlib.pyplot as plt

# Enter File Path here:
file_path = "data_files/data_25_1.dat"

### Extract Data From Dat File - Extract A, b, and F

In [4]:

# Helper Functions:

def generate_x_axis(triangles, y_length): 
    total_min = sum(tri[0] for tri in triangles)
    total_max = sum(tri[2] for tri in triangles)
    x = np.linspace(total_min, total_max, y_length)
    x /= num_experts
    return x  # Calculate total min and max based on start and end points across all triangles

def triangle_function(tri, num_points=1250):
    # Returns un-normalized triangle samples; cumulative_convolve_triangles normalizes by sum (PMF, not density).
    min_val, apex, max_val = tri
    x = np.linspace(min_val, max_val, num_points)
    y = np.maximum(0, np.minimum((x - min_val) / (apex - min_val), (max_val - x) / (max_val - apex)))
    return x, y  # Defines the triangle function

def cumulative_convolve_triangles(triangle_list, num_points=1250):
    # Note: convolution grid is re-linspaced after each np.convolve, assuming uniform dx across triangles.
    # Mixing triangles with widely different widths will distort the density shape.
    if not triangle_list:
        return None, None, None  # In case of empty triangle list
    
    if len(triangle_list) == 1:
        #print('1 encountered')
        x, y = triangle_function(triangle_list[0], num_points=num_points)
        mode_index = np.argmax(y)
        mode_x = x[mode_index]
        y /= np.sum(y) 
        return x, y, mode_x

    # Start with the first triangle
    _, y = triangle_function(triangle_list[0], num_points=num_points)
    total_min = triangle_list[0][0]
    total_max = triangle_list[0][2]
    
    # Convolve with each subsequent triangle
    for tri in triangle_list[1:]:
        _, y2 = triangle_function(tri, num_points=num_points)
        y = np.convolve(y, y2, mode='full')
        
        # Update the range after each convolution
        total_min += tri[0]
        total_max += tri[2]

    y /= np.sum(y)    
    x = np.linspace(total_min, total_max, len(y))  # Adjust x-axis for full range

    # Determine the mode (peak location) after cumulative convolution
    mode_index = np.argmax(y)
    mode_x = x[mode_index]
    return x, y, mode_x



# Main Algorithm
with open(file_path, 'r') as file:
    for i, line in enumerate(file):
        if i == 0:
            data = line.split()
            num_imgs = int(data[0])
            num_experts = int(data[1])
            num_links = int(data[2])
            A_matrix = np.zeros((num_imgs, num_experts))
            F_matrix = [[None for _ in range(num_experts)] for _ in range(num_imgs)]
        
        if i == 2:
            b_array = np.array(line.split()).astype(np.float64)
        
        # From row 4 until the end
        if i >= 3 and i <= (num_links + 2):  # From row 4 until the end
            
            # Split the Data
            data = line.split()

            # Assemble relevant triangle from current row
            start = float(data[3])
            apex = float(data[4])
            end = float(data[5])
            triangle = [start, apex, end]
                      
            # Choose the correct Row for Matrices A and F
            row = (i - 3) // num_experts        

            # Choose the correct column for the given row
            col = int(data[2])
            
            # Initialize Triangle List to 0
            if (i - 3) % num_experts == 0:     
                triangles_dict = {j: [] for j in range(num_experts)}  # Dictionary to store triangles by index
            
            # Append Triangle to corresponding list in the dictionary
            triangles_dict[col].append(triangle)  

            # Perform Convolutions and store to row and column
            if (i - 3 + 1) % num_experts == 0:
                for col_index in range(num_experts):
                    # Get triangles for the current column index
                    triangle_list = triangles_dict[col_index]
                    
                    if triangle_list:  # Only perform convolution if there are triangles
                        x, y, mode_x = cumulative_convolve_triangles(triangle_list)

                        if mode_x is not None:  # Check if mode_x is not None
                            A_matrix[row, col_index] = mode_x/ num_experts                        
                            #F_matrix[row][col_index] = y
                            F_matrix[row][col_index] = (x/num_experts, y)  # Store x, y as a tuple in F_matrix

# Print the rounded matrix
print('Sum of b array', np.sum(b_array))
print(b_array)
print()
print()
print(np.round(A_matrix, 4))

# Plot a sample convolution from F_matrix with dynamically generated x-axis
sample_row = 1
sample_col = 0



# Check if the specific entry in F_matrix is not None
if F_matrix[sample_row][sample_col] is not None:
    sample_x, sample_y = F_matrix[sample_row][sample_col]  # Unpack x and y directly
    
    plt.plot(sample_x, sample_y)
    plt.title(f'Sample Convolution from F_matrix[{sample_row}][{sample_col}]')
    plt.xlabel('x')
    plt.ylabel('Amplitude')
    plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'data_files/prediction_results_25_1.dat'

### Plot the F matrix histograms

In [ ]:
fig, axes = plt.subplots(nrows=len(F_matrix), ncols=len(F_matrix[0]), figsize=(15, 15))

for i in range(len(F_matrix)):
    for j in range(len(F_matrix[i])):
        if F_matrix[i][j] is not None:  # Check if the entry is not None
            x, y = F_matrix[i][j]  # Unpack x and y from F_matrix
            axes[i, j].hist(x, weights=y, bins=50, alpha=0.7, color='blue')  # Use weights for probability distribution

plt.show()


#### Sampling from F Example

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sample from F Function.
def sample_from_F_matrix(F_matrix):
    F_sampled = np.zeros((len(F_matrix), len(F_matrix[0])), dtype=float)
    
    for i in range(len(F_matrix)):
        for j in range(len(F_matrix[i])):
            if F_matrix[i][j] is not None:
                x, y = F_matrix[i][j]  # Unpack x and y
                
                # Sample from x using the probability distribution y / np.sum(y)
                sampled_values = np.random.choice(x, 1, p=y / np.sum(y))
                F_sampled[i, j] = sampled_values
    return F_sampled


# Sample from F_matrix
F_sampled = sample_from_F_matrix(F_matrix)

# Print the rounded sampled matrix
print("F_sampled:")
print(np.round(F_sampled, 4))

##  Define methods for Vertex Finding

This requires installing some libraries such as cdd.  

https://pypi.org/project/pycddlib/  

https://pycddlib.readthedocs.io/en/latest/quickstart.html

you may have to run this command: pip install pycddlib-standalone

In [ ]:
import numpy as np
from scipy.optimize import linprog
import numpy as np
import matplotlib.pyplot as plt
import cdd
import numpy as np

def linearizewithscipy(A,b):

    x_len = A.shape[1]

    # Objective function: Minimize t
    c = np.zeros(x_len + 1)  # n_vars for x and 1 for t
    c[-1] = 1  # Coefficient of t in the objective function

    # Creating Augmented A and Augmented b matrix
    A_ub = np.hstack([A, -np.zeros((A.shape[0], 1))])
    A_ub = np.vstack( [A_ub, np.hstack ([-A, -np.ones((A.shape[0], 1))] )])
#   A_ub = np.vstack( [A_ub, np.hstack ([-A, -np.ones((A.shape[0]-1, 1)),0] )])

    b_ub = np.hstack([b, -b])


    # sum(x) <= 1 (fixed item 3: inequality, not equality)
    A_sum = np.zeros((1, x_len + 1))
    A_sum[0, :x_len] = 1
    b_sum = [1]

    # Bounds for x and t
    bounds = [(0, 1)] * x_len + [(0, None)]  # x in [0, 1] and t in [0, inf]

    # Solving the linear programming problem
    res = linprog(c, A_ub=np.vstack([A_ub, A_sum]), b_ub=np.hstack([b_ub, b_sum]), bounds=bounds, method='highs')  # Fixed (items 2, 3)

    # Extracting the solution
    x = res.x[:-1]
    t = res.x[-1] # Sup norm

    # Testing Results
    b_found = A.dot(x)
    bdifference = b - b_found
    score = bdifference
    norm2 = np.linalg.norm(bdifference, 2)
    print("Optimal value of objective function:", t)
    print("Optimal solution x:", x)
    print()
    print('b desired', b)
    print('b found  ', np.round(b_found,3))
    print()
    print('difference', np.round(score,3))

    print('Sup-norm is:', t)
    print("L2 norm is: ", norm2)

    return t, x, b_found, bdifference 




def montecarlo(array, b_desired):

    # Helper functions
    def initialize_vector():
        k = np.random.randint(5, 20) # Change
        initial_phi = np.zeros(len(array[0]))
        random_indices = np.random.choice(len(initial_phi), k, replace=False)
        numbers = [np.random.uniform(0, 0.1)]
        random_max = 0.05  
        for _ in range(k - 1):
            numbers.append(np.random.uniform(0, random_max - np.sum(numbers)))
        initial_phi[random_indices] = numbers
        return initial_phi

    def calculate_error(phi, penal_mult):
        computed_b = np.dot(array, phi)
        diff = b_desired - computed_b
        penalty1 = -100 * np.sum(diff[diff < 0])  # heuristic surrogate weight
        penalty2 = 0  # .1 * np.sum(diff[diff > 0])  # L1 penalty (currently unused)
        penalty3 = 2 * np.max(diff)  # heuristic surrogate weight
        penalty4 = penal_mult * (1 - sum(phi)) ** 2
        penalties = penalty1 + penalty3 + penalty4 + penalty2
        return penalties
        # Note: penalties above are heuristic surrogate weights — find a good Φ but do not provably minimize sup or L2 norm.

    # Parameters
    T = 10e-4
    T_now = T
    T_min = 1*10e-9
    alpha = 0.7
    num_iterations = 250_000
    penal_mult = 0.1
    step_size = 0.2

    # Initializations
    iter = 0
    counter = []
    error_values = []
    best_values_all = []
    best_phi = np.zeros(len(array[0]))
    best_error = np.inf
    current_phi = np.zeros(len(array[0]))#initialize_vector()
    current_error = np.inf

    # Monte Carlo simulation
    while T > T_min:
        
        if T_now > T:
            print('T',np.round(T,7))
            T_now = T

        for _ in range(num_iterations):
            iter += 1

            # Perturb phi
            new_phi = current_phi.copy()
            index = np.random.randint(0, len(new_phi))
            change = np.random.uniform(-step_size, step_size)
            new_phi[index] += change
            new_phi[new_phi < 0] = 0


            # Skip if sum constraint is violated
            if np.sum(new_phi) > 1:
                continue

            # Calculate new error
            new_error = calculate_error(new_phi, penal_mult)

            # Acceptance criterion
            if new_error < current_error or np.random.uniform() < np.exp((current_error - new_error) / T):
                current_phi = new_phi
                current_error = new_error
                counter.append(iter)
                error_values.append(current_error / 2)  # /2 for plot scaling only
                if new_error < best_error:
                    best_error = new_error
                    best_phi = new_phi
                    best_values_all.append(best_phi)

        # Cooling schedule
        T *= alpha
        step_size *= alpha**(0.5)
        penal_mult *= 1.01

    # Visualization (optional)
    plt.plot(counter[600:], error_values[600:])
    plt.grid()
    plt.show()

    # Results
    b_found = array.dot(best_phi)
    score = b_desired - b_found

    # Print results (optional)
    print('b desired', np.round(b_desired,4))
    print('b found', np.round(b_found, 4))
    print()
    print('difference', np.round(score, 4))
    print("Sup Norm is", np.max(score))
    print("L2 norm is", np.linalg.norm(score, 2))
    print()
    print('best_phi',np.round(best_phi,3))

    return counter, error_values, best_phi, b_found, score


def montecarlo_quiet(array, b_desired):

    # Helper functions
    def initialize_vector():
        k = np.random.randint(5, 20) # Change
        initial_phi = np.zeros(len(array[0]))
        random_indices = np.random.choice(len(initial_phi), k, replace=False)
        numbers = [np.random.uniform(0, 0.1)]
        random_max = 0.05  
        for _ in range(k - 1):
            numbers.append(np.random.uniform(0, random_max - np.sum(numbers)))
        initial_phi[random_indices] = numbers
        return initial_phi

    def calculate_error(phi, penal_mult):
        computed_b = np.dot(array, phi)
        diff = b_desired - computed_b
        penalty1 = -100 * np.sum(diff[diff < 0])
        penalty2 = 0  # .1 * np.sum(diff[diff > 0])  # L1 penalty (currently unused)
        penalty3 = 2 * np.max(diff)
        penalty4 = penal_mult * (1 - sum(phi)) ** 2
        penalties = penalty1 + penalty3 + penalty4 + penalty2
        return penalties

    # Parameters
    T = 10e-4
    T_min = 1*10e-8
    alpha = 0.66
    num_iterations = 250_000
    penal_mult = 0.1
    step_size = 0.3

    # Initializations
    iter = 0
    counter = []
    error_values = []
    best_values_all = []
    best_phi = np.zeros(len(array[0]))
    best_error = np.inf
    current_phi = np.zeros(len(array[0]))#initialize_vector()
    current_error = np.inf

    # Monte Carlo simulation
    while T > T_min:
        

        for _ in range(num_iterations):
            iter += 1

            # Perturb phi
            new_phi = current_phi.copy()
            index = np.random.randint(0, len(new_phi))
            change = np.random.uniform(-step_size, step_size)
            new_phi[index] += change
            new_phi[new_phi < 0] = 0


            # Skip if sum constraint is violated
            if np.sum(new_phi) > 1:
                continue

            # Calculate new error
            new_error = calculate_error(new_phi, penal_mult)

            # Acceptance criterion
            if new_error < current_error or np.random.uniform() < np.exp((current_error - new_error) / T):
                current_phi = new_phi
                current_error = new_error
                counter.append(iter)
                error_values.append(current_error / 2)  # /2 for plot scaling only
                if new_error < best_error:
                    best_error = new_error
                    best_phi = new_phi
                    best_values_all.append(best_phi)

        # Cooling schedule
        T *= alpha
        step_size *= alpha**(0.5)
        penal_mult *= 1.01

    # Visualization (optional)
    #plt.plot(counter[600:], error_values[600:])
    #plt.grid()
    #plt.show()

    # Results
    b_found = array.dot(best_phi)
    score = b_desired - b_found

    # Print results (optional)
    #print('b desired', np.round(b_desired,4))
    #print('b found', np.round(b_found, 4))
    #print()
    #print('difference', np.round(score, 4))
    #print("Sup Norm is", np.max(score))
    #print("L2 norm is", np.linalg.norm(score, 2))
    #print()
    #print('best_phi',np.round(best_phi,3))

    return counter, error_values, best_phi, b_found, score

def vertexcddlist(A, b):
    # Number of variables
    num_vars = A.shape[1]
    
    # Step 1: Add non-negativity constraints
    negative_identity = -np.eye(num_vars)
    
    # Step 2: Add sum constraint
    sum_constraint = np.ones((1, num_vars))
    
    # Stack constraints
    A = np.vstack((A, sum_constraint, negative_identity))
    
    #print('Debugging b', b, b.shape)
    # Step 4: Update b vector
    b = np.hstack((b, 1, np.zeros(num_vars)))  # Append 1 for the sum constraint and zeros for non-negativity
    
    # Step 5: Create inequality representation
    array = np.hstack((b.reshape(-1, 1), -A))
    
    # Create a matrix from the array using inequality representation
    mat = cdd.matrix_from_array(array, rep_type=cdd.RepType.INEQUALITY)
    cdd.matrix_canonicalize(mat)

    # Construct a polyhedron from the matrix
    poly = cdd.polyhedron_from_matrix(mat)

    # Remove redundant constraints
    # poly.remove_redundant()  # Uncomment if needed

    # Copy the generators of the polyhedron
    gen = cdd.copy_generators(poly)

    # Extract the solutions (generators) and ignore the first column
    solutions = np.array(gen.array)
    solutions = solutions[:, 1:]  # Ignore the first column (b values)
    print('Vertices found:', solutions.shape[0], '    Dimension of phi', solutions.shape[1])

    return solutions


def find_max_shortfall_vertex(A,b,solutions):  # uses max(b-Ax) — correct on Ax≤b vertices; see find_supnorm_vertices3 for |·| version
    best_vertices=[]
    best_sup_norm = np.inf
    for boogers in solutions:
        
        sup_norm = np.max((b-A@boogers ))

        if sup_norm == best_sup_norm:
            best_vertices.append(boogers)
        
        if sup_norm < best_sup_norm:
            best_vertices=[]
            best_sup_norm = sup_norm
            best_vertices.append(boogers)

    best_vertices = np.array(best_vertices)

    print()
    print('Max shortfall (signed max b-Ax):', best_sup_norm)
    print('number of best points',(best_vertices.shape[0]))
    print('Example of Best Vertex', best_vertices[0])
    print("Number of active constraints", A.shape[0] +(A.shape[1]-np.count_nonzero( best_vertices[0])))
    print('Sum of example phi',np.sum(best_vertices[0]))
    
    

    return best_vertices, best_sup_norm




# A and b are defined above

def vertexcddlist2(A, b):
    # Number of variables
    num_vars = A.shape[1]
    
    # Step 1: Add non-negativity constraints
    negative_identity = -np.eye(num_vars)
    
    # Step 2: Add sum constraint
    sum_constraint = np.ones((1, num_vars))
    sum_constraint[0, -1] = 0  # exclude last variable (epigraph t) from the simplex Σx≤1 constraint
    #print(sum_constraint)

    
    # Stack constraints
    A = np.vstack((A, sum_constraint, negative_identity))
    
    # Step 4: Update b vector
    b = np.hstack((b, 1, np.zeros(num_vars)))  # Append 1 for the sum constraint and zeros for non-negativity
    
    # Step 5: Create inequality representation
    array = np.hstack((b.reshape(-1, 1), -A))
    
    # Create a matrix from the array using inequality representation
    mat = cdd.matrix_from_array(array, rep_type=cdd.RepType.INEQUALITY)
    cdd.matrix_canonicalize(mat)

    # Construct a polyhedron from the matrix
    poly = cdd.polyhedron_from_matrix(mat)

    # Remove redundant constraints
    #poly.remove_redundant()  # Uncomment if needed

    # Copy the generators of the polyhedron
    gen = cdd.copy_generators(poly)

    # Extract the solutions (generators) and ignore the first column
    solutions = np.array(gen.array)
    solutions = solutions[:, 1:]  # Ignore the first column (b values)
    print('Vertex Walk Complete')
#    print('Vertices found:', solutions.shape[0], ')
    print('Vertices found:', solutions.shape[0], '    Dimension of vertex', solutions.shape[1]-1)

    return solutions



def linear_reformulation(A,b):


    # Creating Augmented A and Augmented b matrix
    A_ub = np.hstack([A, -np.zeros((A.shape[0], 1))])
    A_ub = np.vstack( [A_ub, np.hstack ([-A, -np.ones((A.shape[0], 1))] )])# change this 0 to 1
    b_ub = np.hstack([b, -b])

    return A_ub, b_ub

import numpy as np


def find_supnorm_vertices2(A,b,solutions):

    best_vertices=[]
    best_sup_norm = np.inf
    for boogers in solutions:
    
        sup_norm = np.max((b-A@boogers ))

        if sup_norm == best_sup_norm:
            best_vertices.append(boogers)

        if sup_norm < best_sup_norm:
            best_vertices=[]
            best_sup_norm = sup_norm
            best_vertices.append(boogers)

    best_vertices = np.array(best_vertices)


    print()
    print('Sup norm', best_sup_norm)
    print('number of best points',(best_vertices.shape[0]))
    print('Example of Best Vertex', best_vertices[0,:-1])
    print("Number of active constraints", A.shape[0] +(A.shape[1]-np.count_nonzero( best_vertices[0])))
    print('Sum of example phi',np.sum(best_vertices[0])-best_sup_norm)

    return best_vertices, best_sup_norm

def find_supnorm_vertices3(A, b, solutions):

    best_vertices = []
    best_sup_norm = np.inf
    tolerance = 10e-4  # Set the tolerance here

    for boogers in solutions:
        sup_norm = np.max(np.abs(b - A @ boogers))

        if abs(sup_norm - best_sup_norm) <= tolerance:
            best_vertices.append(boogers)

        elif sup_norm < best_sup_norm:
            best_vertices = []
            best_sup_norm = sup_norm
            best_vertices.append(boogers)

    best_vertices = np.array(best_vertices)

    print()
    print('Sup norm', best_sup_norm)
    print('number of best points', (best_vertices.shape[0]))
    print('Example of Best Vertex', best_vertices[0, :-1])
    print("Number of active constraints", A.shape[0] + (A.shape[1] - np.count_nonzero(best_vertices[0])))
    print('Sum of example phi', np.sum(best_vertices[0]) - best_sup_norm)

    return best_vertices, best_sup_norm


## Algorithm II (Improved) — Epigraph LP + Hit-and-Run

Implementation of the improved inversion algorithm from `Papers/belief-network-inversion-note.html`.

**Inputs**
- `F_bar` — *I × J* mean link-certitude matrix (rows = images, cols = labels)
- `F_sigma` — *I × J* std-dev matrix for link certitudes; pass `None` for deterministic F
- `ell` — *J*-vector of observed labels
- `N` — number of posterior samples (default 500)
- `T` — hit-and-run mixing steps per sample (default 20)

**Returns** `result` dict with keys:
- `M_star` — minimum achievable sup-norm misfit
- `phi_hat` — LP solution (initial point in P*)
- `phi_samples` — *(N × I)* array of hit-and-run draws from P*
- `L_samples` — *(N × J)* array of predicted label vectors
- `L_mean`, `L_std` — posterior mean and std over labels


In [ ]:
from scipy.optimize import linprog
from scipy.stats import truncnorm
import numpy as np


def algorithm_II_improved(F_bar, F_sigma, ell, N=500, T=20):
    F_bar = np.asarray(F_bar, dtype=float)   # (I, J)
    ell   = np.asarray(ell,   dtype=float)   # (J,)
    I, J  = F_bar.shape

    # ------------------------------------------------------------------
    # Phase 2: epigraph LP  ->  M*, phi_hat, P* constraints
    # Variables: [phi (I), t (1)].  Minimise t.
    # Constraint: F_bar.T @ phi + t >= ell  ->  -F_bar.T @ phi - t <= -ell
    # ------------------------------------------------------------------
    c_obj = np.zeros(I + 1); c_obj[-1] = 1.0

    A_ep = np.hstack([-F_bar.T, -np.ones((J, 1))])   # (J, I+1)
    b_ep = -ell

    A_sum = np.zeros((1, I + 1)); A_sum[0, :I] = 1.0
    b_sum = np.array([1.0])

    A_ub = np.vstack([A_ep, A_sum])
    b_ub = np.hstack([b_ep, b_sum])
    bounds = [(0.0, None)] * I + [(0.0, None)]   # phi >= 0, t >= 0

    res = linprog(c_obj, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    if res.status != 0:
        raise RuntimeError(f'LP failed: {res.message}')

    M_star  = float(res.x[-1])
    phi_hat = res.x[:I].copy()

    # P* constraints (without t):
    #   F_bar.T @ phi >= ell - M_star  ->  -F_bar.T @ phi <= -(ell - M_star)
    #   phi >= 0  (enforced via clipping in sampler)
    #   sum(phi) <= 1
    A_pstar = np.vstack([-F_bar.T, np.ones((1, I))])   # (J+1, I)
    b_pstar = np.hstack([-(ell - M_star), 1.0])        # (J+1,)

    # ------------------------------------------------------------------
    # Phase 3+4: hit-and-run sampler on P*, then stochastic forward pass
    # ------------------------------------------------------------------
    def chord_bounds(phi_cur, d):
        # Returns [lam_lo, lam_hi] such that phi_cur + lam*d stays in P*.
        lam_lo, lam_hi = -1e9, 1e9
        Ad  = A_pstar @ d
        rhs = b_pstar - A_pstar @ phi_cur
        for k in range(len(Ad)):
            if Ad[k] > 1e-14:
                lam_hi = min(lam_hi, rhs[k] / Ad[k])
            elif Ad[k] < -1e-14:
                lam_lo = max(lam_lo, rhs[k] / Ad[k])
        # Non-negativity constraints per component
        for i in range(I):
            if d[i] > 1e-14:
                lam_lo = max(lam_lo, -phi_cur[i] / d[i])
            elif d[i] < -1e-14:
                lam_hi = min(lam_hi, -phi_cur[i] / d[i])
        return lam_lo, lam_hi

    phi_samples = np.empty((N, I))
    L_samples   = np.empty((N, J))
    phi_cur     = phi_hat.copy()

    for n in range(N):
        # T hit-and-run mixing steps
        for _ in range(T):
            d = np.random.randn(I)
            d /= np.linalg.norm(d) + 1e-300
            lo, hi = chord_bounds(phi_cur, d)
            if hi > lo + 1e-12:
                lam = np.random.uniform(lo, hi)
                phi_cur = np.clip(phi_cur + lam * d, 0, None)

        phi_samples[n] = phi_cur

        # Sample F_tilde: clipped Gaussian per entry, or use F_bar if no sigma
        if F_sigma is not None:
            s = np.where(np.asarray(F_sigma) < 1e-12, 1e-12, np.asarray(F_sigma))
            a_clip = (0 - F_bar) / s
            b_clip = (1 - F_bar) / s
            F_tilde = truncnorm.rvs(a_clip, b_clip, loc=F_bar, scale=s)
        else:
            F_tilde = F_bar

        L_samples[n] = F_tilde.T @ phi_cur

    return {
        'M_star':      M_star,
        'phi_hat':     phi_hat,
        'phi_samples': phi_samples,
        'L_samples':   L_samples,
        'L_mean':      L_samples.mean(axis=0),
        'L_std':       L_samples.std(axis=0),
    }


In [ ]:
# --- Algorithm II demo (replace with real A_matrix / b_array from the data pipeline) ---
import numpy as np

# Synthetic 3-image, 2-label example
F_bar_demo   = np.array([[0.9, 0.1],
                         [0.5, 0.5],
                         [0.2, 0.8]])
F_sigma_demo = np.full_like(F_bar_demo, 0.05)   # 5% uncertainty on each link
ell_demo     = F_bar_demo.T @ np.array([0.6, 0.0, 0.4])   # exact forward-map label

result = algorithm_II_improved(F_bar_demo, F_sigma_demo, ell_demo, N=500, T=30)

print(f'M* (min sup-norm misfit) = {result["M_star"]:.6f}')
print(f'LP solution phi_hat      = {np.round(result["phi_hat"], 3)}')
print(f'Posterior L mean         = {np.round(result["L_mean"],  3)}')
print(f'Posterior L std          = {np.round(result["L_std"],   3)}')

# To use with the real data pipeline:
#   result = algorithm_II_improved(A_matrix.T, F_sigma, b_array, N=500, T=20)
#   (A_matrix is I x J with rows=images, cols=labels — matches F_bar convention)


#### MonteCarlo Sanity Check for Sup Norm

In [ ]:
counter, error_values, best_phi, b_found, score = montecarlo(A_matrix.T, b_array)

### Finding all Vertices using repeated MonteCarlo Simulations

Generating a few hundred data points

In [ ]:
all_mc_pts = []
number_of_pts = 400

for i in range(number_of_pts):          # a range of 3 takes 2 minutes right now. 
    counter, error_values, best_phi, b_found, score = montecarlo_quiet(A_matrix.T, b_array)
    all_mc_pts.append(best_phi)
    print("round complete")

### keeping the best ones.

In [ ]:

# Find all the sup norms
sup_norms_all_mc_pts = []

for i in range(number_of_pts):
    temp_sup_norm = np.max(b_array-A_matrix.T@all_mc_pts[:][i])
    sup_norms_all_mc_pts.append(temp_sup_norm)

# Sort the indices of 'sup_norms_all_mc_pts' in ascending order
sorted_indices = np.argsort(sup_norms_all_mc_pts)  

# Calculate the number of top points to select (25%)
top_n = int(len(all_mc_pts) * 0.25)  

# Select the indices of the top 25% of 'all_mc_pts'
top_indices = sorted_indices[:top_n]  

# Extract the best Monte Carlo points using the selected indices
the_best_mc_points = [all_mc_pts[i] for i in top_indices]  

the_best_mc_points